In [ ]:
# ============================================================
# Person 4 – Klassifikasjon (Elias)
# Logistisk regresjon (statsmodels) + KNN (scikit-learn)
# Målvariabel: high_rating = 1 hvis Aggregate rating >= 3.5
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (confusion_matrix, roc_curve, auc, accuracy_score)

## Lese og rense data
**Her må Even's datarensing kode**

## Felles train/test-splitt (80/20, `random_state=42`)

In [ ]:
RANDOM_STATE = 42  # felles i gruppen

train_df, test_df = train_test_split(
    df, test_size=0.2, random_state=RANDOM_STATE, stratify=df["high_rating"]
)


## Logistisk regresjon med statsmodels 


In [ ]:
formula = ("high_rating ~ log_cost + log_votes + table_booking "
           "+ online_delivery + C(city_top)")
logit = smf.logit(formula, data=train_df).fit(disp=0)
print(logit.summary())

# Oddsratios med 95 % konfidensintervall  (gjenbruk av np.exp(coef)-ideen)
params = logit.params
conf = logit.conf_int()
odds = pd.DataFrame({
    "OR": np.exp(params),
    "2.5 %": np.exp(conf[0]),
    "97.5 %": np.exp(conf[1]),
    "p-verdi": logit.pvalues,
})
print(odds.round(3))

# Prediksjoner på testsettet
p_logit = logit.predict(test_df)
y_test = test_df["high_rating"].values
y_pred_logit = (p_logit >= 0.5).astype(int)

## KNN med standardisering og kryssvalidering

In [ ]:
features = ["log_cost", "log_votes", "table_booking", "online_delivery"]
X_train_num = train_df[features].values
X_test_num = test_df[features].values

# Standardisering: x_scaled = (x - mean) / std  (samme som i linear_models.py,
# men her via scikit-learn; fit KUN på treningsdata)
scaler = StandardScaler().fit(X_train_num)
X_train_s = scaler.transform(X_train_num)
X_test_s = scaler.transform(X_test_num)

# Velg k med 5-fold kryssvalidering på treningssettet
k_values = range(1, 31, 2)
cv_scores = [
    cross_val_score(KNeighborsClassifier(n_neighbors=k), X_train_s,
                    train_df["high_rating"], cv=5, scoring="accuracy").mean()
    for k in k_values
]
best_k = list(k_values)[int(np.argmax(cv_scores))]
print("Beste k:", best_k, "med CV-nøyaktighet", max(cv_scores).round(3))

plt.figure(figsize=(6, 4))
plt.plot(list(k_values), cv_scores, marker="o")
plt.axvline(best_k, color="red", linestyle="--", label=f"Beste k = {best_k}")
plt.xlabel("k (antall naboer)"); plt.ylabel("CV-nøyaktighet")
plt.title("Valg av k med 5-fold kryssvalidering"); plt.legend(); plt.show()

knn = KNeighborsClassifier(n_neighbors=best_k).fit(X_train_s, train_df["high_rating"])
y_pred_knn = knn.predict(X_test_s)
p_knn = knn.predict_proba(X_test_s)[:, 1]

NameError: name 'train_df' is not defined

## Evaluering: forvirringsmatrise, sensitivitet, spesifisitet

In [ ]:
def evaluate(y_true, y_pred, name):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return {
        "Modell": name,
        "Nøyaktighet": accuracy_score(y_true, y_pred),
        "Sensitivitet": tp / (tp + fn),
        "Spesifisitet": tn / (tn + fp),
    }

results = pd.DataFrame([
    evaluate(y_test, y_pred_logit, "Logistisk regresjon"),
    evaluate(y_test, y_pred_knn, f"KNN (k={best_k})"),
])
print(results.round(3))


## ROC-kurver med AUC 

In [ ]:
# Samme logikk som i problem.ipynb (TPR mot FPR), her via sklearn
plt.figure(figsize=(6, 5))
for p, name in [(p_logit, "Logistisk regresjon"), (p_knn, f"KNN (k={best_k})")]:
    fpr, tpr, _ = roc_curve(y_test, p)
    roc_auc = auc(fpr, tpr)  # tilsvarer np.trapezoid(tpr, fpr)
    plt.plot(fpr, tpr, label=f"{name}, AUC = {roc_auc:.3f}")
plt.plot([0, 1], [0, 1], "--", color="gray", label="Tilfeldig")
plt.xlabel("Falsk positiv rate"); plt.ylabel("Sann positiv rate")
plt.title("ROC-kurver på testsettet"); plt.legend(); plt.grid(True); plt.show()


## Svar på **H2**
> **H2**: *Er det mulig å klassifisere restauranter med høy vurdering (>= 3.5) ut fra pris, popularitet (stemmer), bordbestilling og nettlevering?*

**TODO**: 
- Sammenlign AUC og sensitivitet/spesifisitet for de to modellene og tolk oddsratioene (f.eks. OR for log_votes > 1 => flere stemmer øker oddsene for høy vurdering, alt annet likt).